# 1. Sales Data Analysis

This project analyzes **synthetic sales transaction data** to understand sales performance.

The dataset is made-up practice data (not from a real company). We will use it to learn how to:
- load data with Pandas
- inspect and check data quality
- convert dates
- answer simple business questions about revenue and orders

**Goal:** practice beginner data analytics skills with clean, readable Python.


# 2. Import Libraries

Before we can work with data, we import the tools (libraries) we need:

- **Pandas** (`pd`) - reads CSV files and works with tables of data (rows and columns)
- **NumPy** (`np`) - helps with numbers and calculations (Pandas often uses it behind the scenes)
- **Matplotlib** (`plt`) - draws charts and graphs (we will use this in a later step)

The short names (`pd`, `np`, `plt`) are common nicknames so we type less.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# 3. Load the Dataset

We load the CSV file into a **DataFrame** named `df`.

A DataFrame is Pandas' main table object - think of it like an Excel sheet inside Python.

The path `../data/sales.csv` means:
- start in the `notebooks` folder
- go up one folder (`..`) to the project root
- then open `data/sales.csv`


In [ ]:
# Load the sales CSV into a DataFrame called df
df = pd.read_csv("../data/sales.csv")

# Show the first 5 rows so we can quickly see what the data looks like
df.head()


# 4. Understand the Dataset

Before cleaning or analyzing, we explore the data so we know:
- what columns exist
- how many rows we have
- what types of values are stored
- basic number summaries (mean, min, max, etc.)

This step is part of **EDA** (Exploratory Data Analysis): looking around before drawing conclusions.


### First 5 rows

`df.head()` shows the top of the table. Useful for a quick preview.


In [ ]:
df.head()


### Last 5 rows

`df.tail()` shows the bottom of the table. Helps confirm the file loaded fully.


In [ ]:
df.tail()


### Number of rows and columns

`df.shape` returns `(rows, columns)`.

Example meaning: `(2000, 9)` means 2,000 orders and 9 columns.


In [ ]:
df.shape


### Column names

`df.columns` lists every column name in the DataFrame.


In [ ]:
df.columns


### Data types and non-null counts

`df.info()` shows:
- each column name
- how many non-missing values each column has
- the data type (for example: integer, float, or object/text)


In [ ]:
df.info()


### Basic statistics for numeric columns

`df.describe()` summarizes number columns (Quantity, UnitPrice, Sales, etc.):
- count, mean (average), std (spread)
- min, 25%, 50% (median), 75%, max

This helps you notice unusual values early (for example, a negative price).


In [ ]:
df.describe()


# 5. Check Data Quality

Good analysis starts with checking whether the data is trustworthy.

In this section we check:
- missing values (blank cells)
- duplicate rows (exact copies)
- unique values in important text columns


### Missing values

`df.isnull().sum()` counts how many missing values exist in each column.

- `0` means that column has no missing values
- a number greater than 0 means we may need to investigate later


In [ ]:
df.isnull().sum()


### Duplicate rows

`df.duplicated().sum()` counts how many rows are exact copies of an earlier row.

- `0` means no full duplicate rows were found
- if the number is greater than 0, we would decide carefully whether to remove them later


In [ ]:
df.duplicated().sum()


### Unique values in important columns

`unique()` lists the distinct values in a column.

This helps confirm categories and regions look correct (no unexpected spellings).


In [ ]:
print("Unique categories:")
print(df["Category"].unique())

print("\nUnique regions:")
print(df["Region"].unique())


# 6. Convert OrderDate

Right now, `OrderDate` may be stored as plain text (`object`).

We convert it to a real **datetime** type so Pandas understands it as a date.

**Why this matters:**
- text dates are hard to sort correctly across months/years
- datetime values let us calculate monthly sales, trends, and date ranges later

After converting, we check `df.dtypes` to confirm `OrderDate` is a datetime type.


In [ ]:
# Convert OrderDate from text to datetime
df["OrderDate"] = pd.to_datetime(df["OrderDate"])

# Confirm the data types after conversion
df.dtypes


# 7. Basic Sales Analysis

Now we answer a few starter business questions using simple Pandas calculations.

We are **not** inventing numbers - each result comes directly from `df`.


### Question 1: What is total revenue?

**Business meaning:** How much money did all orders bring in altogether?

We add every value in the `Sales` column with `.sum()`.


In [ ]:
total_revenue = df["Sales"].sum()
print("Total revenue:", total_revenue)


### Question 2: How many orders are there?

**Business meaning:** How many order rows (transactions) are in the dataset?

`len(df)` counts the number of rows.


In [ ]:
number_of_orders = len(df)
print("Number of orders:", number_of_orders)


### Question 3: What is the average sales per order?

**Business meaning:** On average, how much revenue does one order generate?

We use `.mean()` on the `Sales` column.


In [ ]:
average_sales_per_order = df["Sales"].mean()
print("Average sales per order:", average_sales_per_order)


### Question 4: What is the total quantity sold?

**Business meaning:** How many units were sold across all orders?

We add the `Quantity` column with `.sum()`.


In [ ]:
total_quantity_sold = df["Quantity"].sum()
print("Total quantity sold:", total_quantity_sold)


### Question 5: What is the average quantity per order?

**Business meaning:** On average, how many units are in one order?

We use `.mean()` on the `Quantity` column.


In [ ]:
average_quantity_per_order = df["Quantity"].mean()
print("Average quantity per order:", average_quantity_per_order)


# 4. Data Cleaning

**Data cleaning** means checking and fixing problems in your data **before** you trust the analysis results.

Why this matters:
- dirty data can lead to wrong totals and wrong business decisions
- cleaning is not about deleting everything quickly
- first we **check**, then we **decide** what (if anything) needs fixing

In this section we inspect the data carefully.
We will **not** automatically delete rows.


## 1. Check Missing Values

A **missing value** is an empty cell - Pandas often shows it as `NaN` ("Not a Number").

`df.isnull()` checks every cell and returns True where a value is missing.
`.sum()` then counts the missing values in each column.

Why missing values matter:
- they can break totals, averages, and charts
- they may mean something was never recorded

If any column has missing values, we will identify it.
We will **not** delete those rows automatically.


In [ ]:
# Count missing values in each column
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

# Show only columns that have at least one missing value
columns_with_missing = missing_values[missing_values > 0]
print("\nColumns with missing values:")
if len(columns_with_missing) == 0:
    print("None - no missing values were found.")
else:
    print(columns_with_missing)


## 2. Check Duplicate Rows

A **duplicate row** is an exact copy of another row (same values in every column).

Why duplicates matter in sales analysis:
- the same order could be counted twice
- revenue and order counts can become too high

First we count duplicates.
If any exist, we display them before deciding what to do.
We will **not** remove them automatically.


In [ ]:
# Count fully duplicated rows
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

if duplicate_count > 0:
    print("\nDuplicate rows found. Showing them below:")
    display(df[df.duplicated()])
else:
    print("The dataset contains no duplicate rows.")


## 3. Check Data Types

Data types tell Pandas what kind of information each column stores.

Examples:
- numbers for math (`int64`, `float64`)
- dates for time analysis (`datetime64`)
- text for names and labels (`object` or `str`)

Why correct data types matter:
- you cannot reliably add text as if it were money
- dates stored as text are harder to sort and group by month

Expected types for this project:
- `OrderID` - often text (for example `ORD-10001`) or integer, depending on the dataset
- `OrderDate` - datetime
- `Quantity` - numeric
- `UnitPrice` - numeric
- `Sales` - numeric


In [ ]:
# Display the data type of every column
print(df.dtypes)

print("\nQuick type checks:")
print("OrderID type:", df["OrderID"].dtype)
print("OrderDate type:", df["OrderDate"].dtype)
print("Quantity type:", df["Quantity"].dtype)
print("UnitPrice type:", df["UnitPrice"].dtype)
print("Sales type:", df["Sales"].dtype)

# Note for beginners:
# In this dataset, OrderID looks like ORD-10001, so it is stored as text.
# That is okay for an ID label. Quantity, UnitPrice, and Sales should be numeric,
# and OrderDate should already be datetime from the earlier conversion step.


## 4. Validate Quantity

For sales data, Quantity should normally be a positive number (1 or more).

We will check for:
- zero values
- negative values
- missing values

We only **count and show** problems.
We do **not** delete anything automatically.


In [ ]:
# Quantity validation checks
quantity_zero = (df["Quantity"] == 0).sum()
quantity_negative = (df["Quantity"] < 0).sum()
quantity_missing = df["Quantity"].isnull().sum()

print("Quantity equal to 0:", quantity_zero)
print("Quantity less than 0:", quantity_negative)
print("Quantity missing:", quantity_missing)

invalid_quantity_count = quantity_zero + quantity_negative + quantity_missing
print("\nTotal invalid Quantity records:", invalid_quantity_count)

if invalid_quantity_count > 0:
    print("\nRows with invalid Quantity:")
    display(df[(df["Quantity"] <= 0) | (df["Quantity"].isnull())])
else:
    print("No invalid Quantity values were found.")


## 5. Validate UnitPrice

UnitPrice should normally be greater than 0.

We will check for:
- zero values
- negative values
- missing values

Again: check first, do not delete automatically.


In [ ]:
# UnitPrice validation checks
unitprice_zero = (df["UnitPrice"] == 0).sum()
unitprice_negative = (df["UnitPrice"] < 0).sum()
unitprice_missing = df["UnitPrice"].isnull().sum()

print("UnitPrice equal to 0:", unitprice_zero)
print("UnitPrice less than 0:", unitprice_negative)
print("UnitPrice missing:", unitprice_missing)

invalid_unitprice_count = unitprice_zero + unitprice_negative + unitprice_missing
print("\nTotal invalid UnitPrice records:", invalid_unitprice_count)

if invalid_unitprice_count > 0:
    print("\nRows with invalid UnitPrice:")
    display(df[(df["UnitPrice"] <= 0) | (df["UnitPrice"].isnull())])
else:
    print("No invalid UnitPrice values were found.")


## 6. Validate Sales

Sales should normally be greater than 0 for a real purchase row.

We will check for:
- zero values
- negative values
- missing values


In [ ]:
# Sales validation checks
sales_zero = (df["Sales"] == 0).sum()
sales_negative = (df["Sales"] < 0).sum()
sales_missing = df["Sales"].isnull().sum()

print("Sales equal to 0:", sales_zero)
print("Sales less than 0:", sales_negative)
print("Sales missing:", sales_missing)

invalid_sales_count = sales_zero + sales_negative + sales_missing
print("\nTotal invalid Sales records:", invalid_sales_count)

if invalid_sales_count > 0:
    print("\nRows with invalid Sales:")
    display(df[(df["Sales"] <= 0) | (df["Sales"].isnull())])
else:
    print("No invalid Sales values were found.")


## 7. Verify Sales Calculation

This dataset is designed so that:

`Sales = Quantity * UnitPrice`

We create a temporary column `CalculatedSales` to recalculate that formula.
Then we compare it with the original `Sales` column.

Important:
- we are only checking
- we are **not** permanently changing the original `Sales` values


In [ ]:
# Temporary recalculation of Sales
df["CalculatedSales"] = df["Quantity"] * df["UnitPrice"]

# Compare original Sales with CalculatedSales (rounded to 2 decimal places)
df["SalesMatch"] = df["Sales"].round(2) == df["CalculatedSales"].round(2)

print("Do Sales and CalculatedSales match?")
print(df["SalesMatch"].value_counts())

# Interpretation help:
# True  = the row's Sales value matches Quantity * UnitPrice
# False = the row does not match and should be investigated


## 8. Check Date Range

Looking at the earliest and latest OrderDate helps answer:

- What time period does this dataset cover?
- Are the dates spread across about one year, as expected?


In [ ]:
# Earliest and latest order dates
earliest_date = df["OrderDate"].min()
latest_date = df["OrderDate"].max()

print("Earliest OrderDate:", earliest_date)
print("Latest OrderDate:", latest_date)


## 9. Create a Clean DataFrame

If the checks above show no problems that require changes, we create a clean copy:

`clean_df = df.copy()`

`.copy()` makes a separate DataFrame so later analysis can use the cleaned version
without accidentally damaging our working checks.

If real problems had appeared, we would explain them first and only then decide on fixes.
We do **not** silently delete data.


In [ ]:
# Based on the checks above, create a clean working DataFrame.
# We keep helper check columns out of the final clean table.

clean_df = df.drop(columns=["CalculatedSales", "SalesMatch"]).copy()

print("Original df shape:", df.shape)
print("clean_df shape:", clean_df.shape)

# Shape note:
# df may have 2 extra helper columns from the Sales check.
# clean_df keeps the original analysis columns only.
print("\nclean_df columns:")
print(list(clean_df.columns))
print("\nclean_df preview:")
display(clean_df.head())


## 10. Final Data Quality Summary

This summary gathers the main quality checks in one place.

Use it as a quick "health report" for the dataset before deeper analysis.


In [ ]:
# Final data quality summary (using clean_df)
summary = {
    "Number of rows": clean_df.shape[0],
    "Number of columns": clean_df.shape[1],
    "Missing values (total)": int(clean_df.isnull().sum().sum()),
    "Duplicate rows": int(clean_df.duplicated().sum()),
    "Invalid quantities": int(((clean_df["Quantity"] <= 0) | (clean_df["Quantity"].isnull())).sum()),
    "Invalid prices": int(((clean_df["UnitPrice"] <= 0) | (clean_df["UnitPrice"].isnull())).sum()),
    "Invalid sales values": int(((clean_df["Sales"] <= 0) | (clean_df["Sales"].isnull())).sum()),
}

print("DATA QUALITY SUMMARY")
print("--------------------")
for label, value in summary.items():
    print(f"{label}: {value}")


### Data Quality Conclusion

After running the cleaning checks:

- we looked for missing values
- we looked for duplicate rows
- we confirmed important data types
- we validated Quantity, UnitPrice, and Sales
- we verified that Sales matches Quantity * UnitPrice
- we checked the OrderDate range
- we created `clean_df` as our clean working table

If all invalid counts are 0, the dataset is ready for deeper business analysis.

**Stop here for Step 4.**
Next steps later: more business questions and visualizations.


# 5. Business Analysis

The goal of business analysis is to use the cleaned sales data to answer questions about revenue, products, customers, and regions.

In this section we use **Pandas aggregation** and **grouping**:
- **Aggregation** means summarizing many rows into one useful number (for example, total sales).
- **Grouping** means splitting the data into categories first (for example, by Product or Region), then aggregating each group.

We will use `clean_df` from the Data Cleaning section.


## 5.1 Total Revenue

**Business question:** What is the total revenue generated by all sales?

**What total revenue means:**
It is the overall money earned from all orders in the dataset (the sum of the `Sales` column).

**What `.sum()` does:**
`.sum()` adds every value in a column and returns one total.


In [ ]:
# Total revenue = sum of all Sales values
total_revenue = clean_df["Sales"].sum()

print("Total Revenue:", f"${total_revenue:,.2f}")


## 5.2 Number of Orders

**Business question:** How many orders were placed?

We count unique values in `OrderID` with `.nunique()`.

Why counting orders is different from adding quantities:
- **Orders** = how many transactions happened
- **Quantity** = how many units were sold inside those transactions

Example: 1 order can include Quantity = 5. That is still 1 order, but 5 units.


In [ ]:
# Count unique OrderID values
number_of_orders = clean_df["OrderID"].nunique()

print("Number of Orders:", number_of_orders)


## 5.3 Total Units Sold

**Business question:** How many total units were sold?

We add the `Quantity` column.

Difference:
- Number of orders = number of transactions
- Number of units sold = total items sold across all transactions


In [ ]:
# Total units sold = sum of Quantity
total_units_sold = clean_df["Quantity"].sum()

print("Total Units Sold:", total_units_sold)


## 5.4 Average Order Value

**Business question:** What is the average revenue generated per order?

**Average Order Value (AOV)** means:
On average, how much revenue one order brings in.

Formula:

`Average Order Value = Total Revenue / Number of Orders`


In [ ]:
# Average Order Value
average_order_value = total_revenue / number_of_orders

print("Average Order Value:", f"${average_order_value:,.2f}")


## 5.5 Sales by Product

**Business question:** Which products generate the most revenue?

New Pandas ideas used here:

- `groupby("Product")` - split the data into groups, one group per product
- `["Sales"].sum()` - add Sales inside each group
- `sort_values(ascending=False)` - sort so the highest revenue appears first

Simple example:
If Laptop has 3 rows with Sales 100, 200, and 50, then Laptop total = 350.


In [ ]:
# Total Sales for each product, highest first
sales_by_product = (
    clean_df.groupby("Product")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Sales by Product:")
print(sales_by_product)


## 5.6 Sales by Category

**Business question:** Which product category generates the most revenue?

Business meaning:
This shows which category (for example Electronics or Furniture) contributes the most money.
A business might invest more marketing or inventory in stronger categories.


In [ ]:
# Total Sales for each category, highest first
sales_by_category = (
    clean_df.groupby("Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Sales by Category:")
print(sales_by_category)


## 5.7 Sales by Region

**Business question:** Which region generates the most revenue?

This helps a business compare performance across East, West, North, and South.


In [ ]:
# Total Sales for each region, highest first
sales_by_region = (
    clean_df.groupby("Region")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

print("Sales by Region:")
print(sales_by_region)


## 5.8 Monthly Sales

**Business question:** How do sales change from month to month?

Datetime operation used here:
- `.dt.to_period("M")` extracts year-month from `OrderDate` (for example, 2024-01)

We then group by that month and sum Sales.

Why chronological sorting matters:
If months are sorted alphabetically, April can appear before February.
Sorting with `.sort_index()` keeps months in real calendar order (Jan, Feb, Mar, ...).


In [ ]:
# Create a year-month label from OrderDate, then total Sales per month
monthly_sales = (
    clean_df.groupby(clean_df["OrderDate"].dt.to_period("M"))["Sales"]
    .sum()
    .sort_index()  # chronological order
)

print("Monthly Sales:")
print(monthly_sales)


## 5.9 Highest Sales Month

**Business question:** Which month generated the highest revenue?

We use `.idxmax()` and `.max()` on the monthly sales result:
- `.idxmax()` returns the month label with the highest value
- `.max()` returns that highest sales number

We do not type the answer manually.


In [ ]:
# Find the month with the highest Sales
highest_sales_month = monthly_sales.idxmax()
highest_month_revenue = monthly_sales.max()

print("Highest Sales Month:", highest_sales_month)
print("Revenue:", f"${highest_month_revenue:,.2f}")


## 5.10 Top 10 Customers

**Business question:** Which customers generate the most revenue?

We group by `CustomerName`, sum Sales, sort highest-first, then use `.head(10)`.

`.head(10)` means: show only the first 10 rows after sorting.

Why this is useful:
Businesses can identify high-value customers for loyalty programs, follow-ups, or account management.


In [ ]:
# Top 10 customers by revenue
top_10_customers = (
    clean_df.groupby("CustomerName")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top 10 Customers by Revenue:")
print(top_10_customers)


## 5.11 Top Products by Units Sold

**Business question:** Which products sold the most units?

Difference to remember:
- Best by **revenue** = products that made the most money
- Best by **units sold** = products that sold the most quantity

A cheap product can sell many units but still make less revenue than an expensive product.


In [ ]:
# Top 10 products by total Quantity sold
top_products_by_units = (
    clean_df.groupby("Product")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

print("Top Products by Units Sold:")
print(top_products_by_units)


## 5.12 Category Revenue Percentage

**Business question:** What percentage of total revenue comes from each category?

Formula:

`category revenue / total revenue * 100`

This helps a business see revenue distribution:
for example, whether one category dominates sales or revenue is more balanced.

Percentages should add up to about 100% (tiny differences can happen because of rounding).


In [ ]:
# Revenue percentage by category
category_revenue = clean_df.groupby("Category")["Sales"].sum()
category_revenue_pct = (category_revenue / total_revenue) * 100

# Sort highest percentage first and round for readability
category_revenue_pct = category_revenue_pct.sort_values(ascending=False).round(2)

print("Category Revenue Percentage:")
print(category_revenue_pct)
print("\nSum of percentages:", round(category_revenue_pct.sum(), 2))


## 5.13 Create a Business Summary Table

This table collects the key metrics in one place.

Every value is calculated from `clean_df` (nothing is typed in by hand).


In [ ]:
# Build a simple summary table from calculated results
summary_data = {
    "Metric": [
        "Total Revenue",
        "Number of Orders",
        "Total Units Sold",
        "Average Order Value",
        "Highest Revenue Product",
        "Highest Revenue Category",
        "Highest Revenue Region",
        "Highest Revenue Month",
    ],
    "Value": [
        f"${total_revenue:,.2f}",
        number_of_orders,
        total_units_sold,
        f"${average_order_value:,.2f}",
        f"{sales_by_product.idxmax()} (${sales_by_product.max():,.2f})",
        f"{sales_by_category.idxmax()} (${sales_by_category.max():,.2f})",
        f"{sales_by_region.idxmax()} (${sales_by_region.max():,.2f})",
        f"{highest_sales_month} (${highest_month_revenue:,.2f})",
    ],
}

business_summary = pd.DataFrame(summary_data)
print("Business Summary Table")
print("----------------------")
display(business_summary)


## Business Insights

These observations come **only** from the calculations above (not invented numbers):

- The highest-revenue category was **Electronics** ($623,514.64).
- The highest-revenue region was **North** ($271,444.12).
- The highest-revenue product was **Laptop** ($457,297.40).
- The strongest sales month was **2024-02** ($100,182.04).
- The top customer was **Sam Patel**, generating $70,708.34.
- The product with the most units sold was **Keyboard** (715 units).
- Total revenue was $1,038,438.39 across 2,000 orders.
- Average Order Value was $519.22.

Note: best product by **revenue** may differ from best product by **units sold**,
because expensive items can generate more money even if fewer units are sold.


# Stop Here (Step 5)

You have finished the Business Analysis section.

Next steps later (not now):
- visualizations with Matplotlib
- SQL queries
- README documentation

Before moving on, re-read your printed results and ask:
Do these numbers look reasonable for about 2,000 sales rows?


# 6. Data Visualization

Data visualization helps us understand patterns and trends in the sales data more easily than looking at tables alone.

In this section we create charts for:
- Monthly sales
- Category sales
- Regional sales
- Top products
- Top customers
- Units sold by category

We use **Matplotlib** only.

### Quick chart types (beginner guide)

- **Line chart** (`plt.plot`) - best for change over time (example: monthly sales)
- **Bar chart** (`plt.bar`) - best for comparing categories side by side
- **Horizontal bar chart** (`plt.barh`) - useful when labels are long (product/customer names)

### Common Matplotlib commands we will use

- `plt.figure(figsize=(width, height))` - create a new chart canvas
- `plt.plot()` / `plt.bar()` / `plt.barh()` - draw the chart
- `plt.title()` - chart title
- `plt.xlabel()` / `plt.ylabel()` - axis labels
- `plt.xticks(rotation=...)` - tilt x-axis labels so they are readable
- `plt.tight_layout()` - reduce overlapping text
- `plt.savefig(...)` - save the chart as an image file
- `plt.show()` - display the chart in the notebook
- `plt.close()` - close the figure after saving/showing


## Setup: charts folder

Before creating charts, we make sure the output folder exists:

`output/charts/`

All chart image files will be saved there.


In [ ]:
from pathlib import Path

# Make sure the charts folder exists
charts_folder = Path("../output/charts")
charts_folder.mkdir(parents=True, exist_ok=True)

print("Charts will be saved to:", charts_folder.resolve())


## 6.1 Monthly Sales Trend

We create a **line chart** of total Sales by month.

Steps:
1. Group `clean_df` by month from `OrderDate`
2. Sort months chronologically
3. Plot with `plt.plot(..., marker="o")` so each month has a marker point


In [ ]:
# Monthly sales (chronological order)
monthly_sales_chart = (
    clean_df.groupby(clean_df["OrderDate"].dt.to_period("M"))["Sales"]
    .sum()
    .sort_index()
)

# Convert period labels to text so Matplotlib can plot them easily
months = monthly_sales_chart.index.astype(str)
sales_values = monthly_sales_chart.values

# Create the figure (canvas size in inches: width=10, height=5)
plt.figure(figsize=(10, 5))

# Line chart with markers
plt.plot(months, sales_values, marker="o")

plt.title("Monthly Sales Trend", fontsize=14)
plt.xlabel("Month", fontsize=12)
plt.ylabel("Sales ($)", fontsize=12)
plt.xticks(rotation=45)

# Simple currency-style formatting on the y-axis
plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

plt.tight_layout()
plt.savefig(charts_folder / "monthly_sales.png")
plt.show()
plt.close()

print("Saved: output/charts/monthly_sales.png")


### What this chart shows

The Monthly Sales Trend chart shows how total revenue changes from month to month.

- Rising points mean sales increased that month
- Falling points mean sales decreased that month

Look at the line shape to spot stronger and weaker periods.
Use the calculated monthly table from Step 5 to confirm exact values.


## 6.2 Sales by Category

We create a **bar chart** comparing total Sales across categories.

`plt.bar(x, height)` draws vertical bars.


In [ ]:
# Sales by category
sales_by_category_chart = (
    clean_df.groupby("Category")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(sales_by_category_chart.index, sales_by_category_chart.values)

plt.title("Sales by Category", fontsize=14)
plt.xlabel("Category", fontsize=12)
plt.ylabel("Sales ($)", fontsize=12)
plt.xticks(rotation=30)

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

plt.tight_layout()
plt.savefig(charts_folder / "sales_by_category.png")
plt.show()
plt.close()

print("Saved: output/charts/sales_by_category.png")


### What this chart shows

This chart compares revenue by product category.

Taller bars mean higher total Sales for that category.


## 6.3 Sales by Region

We create another **bar chart**, this time for regions.


In [ ]:
# Sales by region
sales_by_region_chart = (
    clean_df.groupby("Region")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(sales_by_region_chart.index, sales_by_region_chart.values)

plt.title("Sales by Region", fontsize=14)
plt.xlabel("Region", fontsize=12)
plt.ylabel("Sales ($)", fontsize=12)

plt.gca().yaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

plt.tight_layout()
plt.savefig(charts_folder / "sales_by_region.png")
plt.show()
plt.close()

print("Saved: output/charts/sales_by_region.png")


### What this chart shows

This chart compares total revenue across regions (East, West, North, South).

It helps you quickly see which region contributed more sales dollars.


## 6.4 Top 10 Products by Revenue

We use a **horizontal bar chart** (`plt.barh`) because product names can be long.

Tip for readability:
Sort values ascending before `barh`, so the highest revenue product appears at the top.


In [ ]:
# Top 10 products by revenue
top_products_chart = (
    clean_df.groupby("Product")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values(ascending=True)  # for horizontal chart readability
)

plt.figure(figsize=(10, 6))
plt.barh(top_products_chart.index, top_products_chart.values)

plt.title("Top 10 Products by Revenue", fontsize=14)
plt.xlabel("Sales ($)", fontsize=12)
plt.ylabel("Product", fontsize=12)

plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

plt.tight_layout()
plt.savefig(charts_folder / "top_products.png")
plt.show()
plt.close()

print("Saved: output/charts/top_products.png")


### What this chart shows

This chart ranks products by revenue.

Longer bars mean higher total Sales for that product.


## 6.5 Top 10 Customers by Revenue

Another **horizontal bar chart**, now for customers.


In [ ]:
# Top 10 customers by revenue
top_customers_chart = (
    clean_df.groupby("CustomerName")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values(ascending=True)  # highest at the top
)

plt.figure(figsize=(10, 6))
plt.barh(top_customers_chart.index, top_customers_chart.values)

plt.title("Top 10 Customers by Revenue", fontsize=14)
plt.xlabel("Sales ($)", fontsize=12)
plt.ylabel("Customer", fontsize=12)

plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

plt.tight_layout()
plt.savefig(charts_folder / "top_customers.png")
plt.show()
plt.close()

print("Saved: output/charts/top_customers.png")


### What this chart shows

This chart highlights the customers who generated the most revenue.

It is useful for spotting high-value customer relationships.


## 6.6 Units Sold by Category

This **bar chart** uses `Quantity` instead of `Sales`.

That means we are looking at sales **volume** (units), not revenue (dollars).


In [ ]:
# Units sold by category
units_by_category_chart = (
    clean_df.groupby("Category")["Quantity"]
    .sum()
    .sort_values(ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(units_by_category_chart.index, units_by_category_chart.values)

plt.title("Units Sold by Category", fontsize=14)
plt.xlabel("Category", fontsize=12)
plt.ylabel("Units Sold", fontsize=12)
plt.xticks(rotation=30)

plt.tight_layout()
plt.savefig(charts_folder / "units_by_category.png")
plt.show()
plt.close()

print("Saved: output/charts/units_by_category.png")


### What this chart shows

This chart compares how many units were sold in each category.

A category can lead in units sold but not lead in revenue, if its products are cheaper.


## Visualization Summary

Here is what each chart helps us understand:

1. **Monthly Sales Trend** → how sales change over time
2. **Sales by Category** → category performance by revenue
3. **Sales by Region** → regional performance by revenue
4. **Top Products** → highest revenue products
5. **Top Customers** → highest revenue customers
6. **Units Sold by Category** → sales volume by category

These charts support the tables from Step 5. Always compare chart shapes with the calculated numbers.


# Stop Here (Step 6)

You have finished the Data Visualization section.

Saved chart files should appear in:

`output/charts/`

- monthly_sales.png
- sales_by_category.png
- sales_by_region.png
- top_products.png
- top_customers.png
- units_by_category.png

Next steps later (not now): SQL queries and README documentation.
